# 🛒 Smart Customer Behavior & Churn Prediction System
## Complete Big Data Pipeline Project

---

### 📋 Problem Statement

E-commerce platforms face critical challenges:
- **Cart Abandonment**: 70% of users add items but don't complete purchase
- **Customer Churn**: Users browse extensively but never convert
- **Revenue Loss**: Billions lost annually due to lack of retention strategies

### 🎯 Solution

Build a **scalable Big Data pipeline** that:
1. Processes millions of behavioral events in real-time
2. Predicts churn risk using Machine Learning
3. Generates actionable retention strategies (discounts, reminders)

### 🏗️ Big Data Architecture

```
┌─────────────┐
│  Raw CSV    │
└──────┬──────┘
       │
       ▼
┌─────────────────────┐
│  Layer 1: Ingestion │ ← PySpark
└──────┬──────────────┘
       │
       ▼
┌──────────────────────────┐
│  Layer 2: Raw Storage    │ ← Parquet (partitioned)
└──────┬───────────────────┘
       │
       ▼
┌─────────────────────────────┐
│  Layer 3: Quality Check     │ ← Null/Duplicates/Stats
└──────┬──────────────────────┘
       │
       ▼
┌──────────────────────────────┐
│  Layer 4: Data Cleaning      │ ← Handle nulls/types
└──────┬───────────────────────┘
       │
       ▼
┌───────────────────────────────────┐
│  Layer 5: Feature Engineering     │ ← User aggregations
└──────┬────────────────────────────┘
       │
       ▼
┌────────────────────────────────────┐
│  Layer 6: Churn Logic & Labels     │ ← Risk classification
└──────┬─────────────────────────────┘
       │
       ▼
┌────────────────────────────────┐
│  Layer 7: Processed Storage    │ ← Parquet (features)
└──────┬─────────────────────────┘
       │
       ▼
┌───────────────────────────┐
│  Layer 8: Analytics       │ ← Stats tables
└──────┬────────────────────┘
       │
       ▼
┌──────────────────────────────┐
│  Layer 9: Machine Learning   │ ← LR, DT, RF models
└──────┬───────────────────────┘
       │
       ▼
┌──────────────────────────────┐
│  Layer 10: Decision Engine   │ ← Action recommendations
└──────┬───────────────────────┘
       │
       ▼
┌──────────────────────────┐
│  Layer 11: CSV Exports   │ ← Dashboard feeds
└──────────────────────────┘
```

### 📊 Dataset

**eCommerce Behavior Dataset** from Kaggle
- **Source**: Multi-category online store
- **Size**: 100M+ events
- **Period**: Multiple months

**Columns**:
- `event_time`: Timestamp
- `event_type`: view, cart, remove_from_cart, purchase
- `product_id`, `category_id`, `category_code`, `brand`, `price`
- `user_id`, `user_session`

---

## 0️⃣ Environment Setup

In [ ]:
# Install dependencies
!pip install -q pyspark==3.5.1 pyarrow pandas matplotlib seaborn
print('✅ Packages installed successfully')

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Dataset path (upload to Colab first)
CSV_PATH = '2019-Oct.csv'  # Or '2019-Nov.csv' - from Kaggle dataset

if not os.path.exists(CSV_PATH):
    print('⚠️  Upload dataset first:')
    print('   1. Download from: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store')
    print('   2. Upload 2019-Oct.csv via Files → Upload')
    print('   3. Re-run this cell')
else:
    import pandas as pd
    preview = pd.read_csv(CSV_PATH, nrows=5)
    print(f'✅ Dataset found: {CSV_PATH}')
    print(f'Columns ({len(preview.columns)}): {list(preview.columns)}')
    print(f'\n📊 Preview:')
    print(preview)

---
## Layer 1: Data Ingestion 📥

Load raw data using PySpark with optimized configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Initialize Spark with Big Data optimizations
spark = (SparkSession.builder
         .appName('EcommerceChurnPrediction')
         .master('local[*]')  # Use all CPU cores
         .config('spark.driver.memory', '4g')
         .config('spark.executor.memory', '4g')
         .config('spark.sql.shuffle.partitions', '8')  # Optimize for joins/aggregations
         .config('spark.default.parallelism', '8')
         .config('spark.ui.enabled', 'false')  # Disable UI in Colab
         .getOrCreate())

spark.sparkContext.setLogLevel('ERROR')  # Reduce log noise

print('✅ Spark initialized successfully')
print(f'   Version: {spark.version}')
print(f'   Available cores: {spark.sparkContext.defaultParallelism}')
print(f'   Driver memory: 4g')
print(f'   Shuffle partitions: 8')

In [ ]:
# Define schema for type safety and faster loading
# This prevents Spark from inferring types (which requires scanning entire file)
ecommerce_schema = StructType([
    StructField('event_time', StringType(), True),
    StructField('event_type', StringType(), True),
    StructField('product_id', LongType(), True),
    StructField('category_id', LongType(), True),
    StructField('category_code', StringType(), True),
    StructField('brand', StringType(), True),
    StructField('price', FloatType(), True),
    StructField('user_id', LongType(), True),
    StructField('user_session', StringType(), True)
])

print('✅ Schema defined')
print('\n📋 Expected columns and types:')
for field in ecommerce_schema.fields:
    print(f'   {field.name:20s} → {str(field.dataType):15s}')

In [ ]:
# Load data with sampling for development
# Note: In production, remove .sample() to process full dataset

print('📂 Loading data...')
print('   [DEV MODE] Using 10% sample for faster iteration')
print('   [PRODUCTION] Remove .sample() to process full dataset\n')

df_raw = (spark.read
          .format('csv')
          .option('header', 'true')
          .option('inferSchema', 'false')  # Use our predefined schema
          .schema(ecommerce_schema)
          .load(CSV_PATH))

# Sample 10% for development (comment out for production)
df_raw = df_raw.sample(fraction=0.1, seed=42)

# Cache for faster repeated access
df_raw.cache()

# Get statistics
row_count = df_raw.count()
col_count = len(df_raw.columns)
partitions = df_raw.rdd.getNumPartitions()

print(f'✅ Data loaded successfully')
print(f'   Total rows: {row_count:,}')
print(f'   Total columns: {col_count}')
print(f'   Partitions: {partitions}')
print(f'   Memory: Cached in RAM for fast access')

In [ ]:
# Inspect schema and sample data
print('=' * 60)
print('SCHEMA')
print('=' * 60)
df_raw.printSchema()

print('\n' + '=' * 60)
print('SAMPLE DATA (5 rows)')
print('=' * 60)
df_raw.show(5, truncate=False, vertical=False)

---
## Layer 2: Raw Storage (Parquet) 💾

Convert CSV → Parquet with partitioning for optimized querying

In [ ]:
# Create storage hierarchy
BASE_PATH = '/content/ecommerce_bigdata'
RAW_STORAGE = f'{BASE_PATH}/raw'
PROCESSED_STORAGE = f'{BASE_PATH}/processed'
ANALYTICS_STORAGE = f'{BASE_PATH}/analytics'
ML_STORAGE = f'{BASE_PATH}/ml_models'
EXPORT_STORAGE = f'{BASE_PATH}/exports'

for path in [RAW_STORAGE, PROCESSED_STORAGE, ANALYTICS_STORAGE, ML_STORAGE, EXPORT_STORAGE]:
    os.makedirs(path, exist_ok=True)

print('✅ Storage directories created')
print(f'   Base path: {BASE_PATH}')
print(f'   ├── raw/         (Parquet - partitioned by event_type)')
print(f'   ├── processed/   (Parquet - user features)')
print(f'   ├── analytics/   (Parquet + CSV tables)')
print(f'   ├── ml_models/   (Trained models)')
print(f'   └── exports/     (CSV for dashboards)')

In [ ]:
# Save as Parquet with partitioning by event_type
# Partitioning benefits:
#   - Faster queries when filtering by event_type
#   - Better compression (similar data grouped together)
#   - Enables partition pruning (skip irrelevant files)

print('💾 Converting CSV → Parquet (partitioned by event_type)...')
print('   This enables partition pruning for faster queries\n')

(df_raw.write
 .mode('overwrite')
 .partitionBy('event_type')  # Create separate folders per event_type
 .parquet(f'{RAW_STORAGE}/events'))

print('✅ Parquet files written')

# Verify partitions
!ls -lh {RAW_STORAGE}/events/

print('\n📊 Partition structure:')
!du -sh {RAW_STORAGE}/events/event_type=*/ 2>/dev/null || echo "  Folders created per event_type"

In [ ]:
# Compare CSV vs Parquet
import subprocess

# Get CSV size
csv_size_bytes = os.path.getsize(CSV_PATH)
csv_size_mb = csv_size_bytes / (1024 * 1024)

# Get Parquet size
result = subprocess.run(['du', '-sb', f'{RAW_STORAGE}/events'], 
                       capture_output=True, text=True)
parquet_size_bytes = int(result.stdout.split()[0])
parquet_size_mb = parquet_size_bytes / (1024 * 1024)

# Calculate compression ratio
compression_ratio = (1 - parquet_size_bytes / csv_size_bytes) * 100

print('=' * 60)
print('CSV vs PARQUET COMPARISON')
print('=' * 60)
print(f'CSV size:              {csv_size_mb:>10.2f} MB')
print(f'Parquet size:          {parquet_size_mb:>10.2f} MB')
print(f'Space saved:           {compression_ratio:>10.1f}%')
print()
print('Why Parquet is better for Big Data:')
print('  ✓ Columnar storage → faster analytics (read only needed columns)')
print('  ✓ Built-in compression → 50-80% smaller than CSV')
print('  ✓ Schema embedded → type safety, no parsing errors')
print('  ✓ Partitioning support → query only relevant data')
print('  ✓ Predicate pushdown → filter at file level (skip entire files)')
print('=' * 60)

---
## Layer 3: Data Quality Check 🔍

Analyze data quality before processing

In [ ]:
# Read from Parquet (faster than CSV)
print('📂 Loading data from Parquet...')
df = spark.read.parquet(f'{RAW_STORAGE}/events')
df.cache()

total_rows = df.count()
print(f'✅ Loaded {total_rows:,} rows from Parquet')

In [ ]:
# Check for null values in each column
print('=' * 60)
print('NULL VALUE ANALYSIS')
print('=' * 60)

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df.columns
]).collect()[0].asDict()

print(f'{"Column":<20} {"Null Count":>12} {"Null %":>10}')
print('-' * 60)

for col_name, null_count in null_counts.items():
    null_pct = (null_count / total_rows) * 100
    status = '⚠️ ' if null_pct > 10 else '  '
    print(f'{status}{col_name:<20} {null_count:>12,} {null_pct:>9.2f}%')

print('\n💡 Columns with >10% nulls need special handling')

In [ ]:
# Check for duplicate rows
print('=' * 60)
print('DUPLICATE ANALYSIS')
print('=' * 60)

duplicate_count = total_rows - df.dropDuplicates().count()
duplicate_pct = (duplicate_count / total_rows) * 100

print(f'Total rows:        {total_rows:>12,}')
print(f'Duplicate rows:    {duplicate_count:>12,}')
print(f'Duplicate %:       {duplicate_pct:>11.2f}%')

if duplicate_pct < 1:
    print('✅ Low duplication - acceptable')
else:
    print('⚠️  High duplication - will remove in cleaning step')

In [ ]:
# Distribution of event_type
print('=' * 60)
print('EVENT TYPE DISTRIBUTION')
print('=' * 60)

event_dist = (df.groupBy('event_type')
              .count()
              .withColumn('percentage', F.round((F.col('count') / total_rows) * 100, 2))
              .orderBy(F.desc('count')))

event_dist.show(truncate=False)

print('\n💡 Insights:')
print('   - "view" events dominate (browsing behavior)')
print('   - "purchase" events are minority (conversion funnel)')
print('   - Gap between cart and purchase = abandonment opportunity')

In [ ]:
# Price statistics
print('=' * 60)
print('PRICE STATISTICS')
print('=' * 60)

price_stats = df.select(
    F.count('price').alias('non_null_count'),
    F.min('price').alias('min_price'),
    F.max('price').alias('max_price'),
    F.mean('price').alias('avg_price'),
    F.expr('percentile_approx(price, 0.5)').alias('median_price'),
    F.stddev('price').alias('std_price')
).collect()[0]

print(f'Non-null prices:   {price_stats.non_null_count:>12,}')
print(f'Min price:         ${price_stats.min_price:>11,.2f}')
print(f'Max price:         ${price_stats.max_price:>11,.2f}')
print(f'Average price:     ${price_stats.avg_price:>11,.2f}')
print(f'Median price:      ${price_stats.median_price:>11,.2f}')
print(f'Std deviation:     ${price_stats.std_price:>11,.2f}')

# Check for outliers
if price_stats.max_price > price_stats.avg_price * 10:
    print('\n⚠️  Potential outliers detected (very high prices)')

In [ ]:
# User activity summary
print('=' * 60)
print('USER ACTIVITY SUMMARY')
print('=' * 60)

user_stats = df.select(
    F.countDistinct('user_id').alias('unique_users'),
    F.countDistinct('user_session').alias('unique_sessions'),
    F.countDistinct('product_id').alias('unique_products'),
    F.countDistinct('category_id').alias('unique_categories'),
    F.countDistinct('brand').alias('unique_brands')
).collect()[0]

print(f'Unique users:      {user_stats.unique_users:>12,}')
print(f'Unique sessions:   {user_stats.unique_sessions:>12,}')
print(f'Unique products:   {user_stats.unique_products:>12,}')
print(f'Unique categories: {user_stats.unique_categories:>12,}')
print(f'Unique brands:     {user_stats.unique_brands:>12,}')

avg_events_per_user = total_rows / user_stats.unique_users
avg_sessions_per_user = user_stats.unique_sessions / user_stats.unique_users

print(f'\nAvg events/user:   {avg_events_per_user:>12.1f}')
print(f'Avg sessions/user: {avg_sessions_per_user:>12.1f}')

---
## Layer 4: Data Cleaning 🧹

Handle nulls, fix types, remove invalids

In [ ]:
print('🧹 Starting data cleaning...')
print(f'   Initial rows: {df.count():,}\n')

# Step 1: Convert event_time to timestamp
print('1️⃣ Converting event_time to timestamp...')
df_clean = df.withColumn(
    'event_timestamp',
    F.to_timestamp('event_time', 'yyyy-MM-dd HH:mm:ss z')
)

# If format doesn't match, try alternative
if df_clean.filter(F.col('event_timestamp').isNull()).count() > 0:
    print('   Trying alternative timestamp format...')
    df_clean = df.withColumn(
        'event_timestamp',
        F.to_timestamp('event_time')
    )

print('   ✅ Timestamp conversion complete')

# Step 2: Fill nulls in brand and category_code
print('\n2️⃣ Handling missing values...')
df_clean = df_clean.fillna({
    'brand': 'unknown',
    'category_code': 'unknown'
})
print('   ✅ Filled nulls: brand, category_code → "unknown"')

# Step 3: Remove rows with critical nulls
print('\n3️⃣ Removing rows with critical nulls...')
before_count = df_clean.count()

df_clean = df_clean.filter(
    F.col('user_id').isNotNull() &
    F.col('product_id').isNotNull() &
    F.col('event_type').isNotNull() &
    F.col('event_timestamp').isNotNull()
)

after_count = df_clean.count()
removed = before_count - after_count
print(f'   Removed {removed:,} rows with critical nulls')
print(f'   Remaining: {after_count:,} rows')

# Step 4: Remove invalid prices
print('\n4️⃣ Filtering invalid prices...')
before_count = df_clean.count()

df_clean = df_clean.filter(
    (F.col('price').isNotNull()) &
    (F.col('price') > 0) &
    (F.col('price') < 100000)  # Remove extreme outliers
)

after_count = df_clean.count()
removed = before_count - after_count
print(f'   Removed {removed:,} rows with invalid prices')
print(f'   Remaining: {after_count:,} rows')

# Step 5: Remove duplicates
print('\n5️⃣ Removing duplicate rows...')
before_count = df_clean.count()

df_clean = df_clean.dropDuplicates()

after_count = df_clean.count()
removed = before_count - after_count
print(f'   Removed {removed:,} duplicate rows')
print(f'   Final: {after_count:,} rows')

# Step 6: Drop original event_time (keep timestamp version)
print('\n6️⃣ Dropping redundant columns...')
df_clean = df_clean.drop('event_time')
print('   ✅ Dropped: event_time (kept event_timestamp)')

# Cache cleaned data
df_clean.cache()

print('\n' + '=' * 60)
print('CLEANING SUMMARY')
print('=' * 60)
print(f'Initial rows:  {df.count():>12,}')
print(f'Final rows:    {df_clean.count():>12,}')
print(f'Rows removed:  {df.count() - df_clean.count():>12,}')
print(f'Retention:     {(df_clean.count()/df.count())*100:>11.1f}%')
print('=' * 60)

In [ ]:
# Verify cleaned data
print('📊 Cleaned data schema:')
df_clean.printSchema()

print('\n📋 Sample cleaned data:')
df_clean.show(5, truncate=False)

---
## Layer 5: Feature Engineering 🔧

**CRITICAL LAYER** - Aggregate user behavior into meaningful features

In [ ]:
print('🔧 Starting Feature Engineering...')
print('   Creating user-level aggregations\n')

# User-level aggregations - the foundation for churn prediction
user_features = df_clean.groupBy('user_id').agg(
    # Event counts
    F.sum(F.when(F.col('event_type') == 'view', 1).otherwise(0)).alias('view_count'),
    F.sum(F.when(F.col('event_type') == 'cart', 1).otherwise(0)).alias('cart_count'),
    F.sum(F.when(F.col('event_type') == 'remove_from_cart', 1).otherwise(0)).alias('remove_from_cart_count'),
    F.sum(F.when(F.col('event_type') == 'purchase', 1).otherwise(0)).alias('purchase_count'),
    
    # Financial metrics
    F.sum(F.when(F.col('event_type') == 'purchase', F.col('price')).otherwise(0)).alias('total_spent'),
    F.avg('price').alias('average_price'),
    F.max('price').alias('max_price_viewed'),
    F.min('price').alias('min_price_viewed'),
    
    # Diversity metrics
    F.countDistinct('product_id').alias('unique_products'),
    F.countDistinct('category_id').alias('unique_categories'),
    F.countDistinct('brand').alias('unique_brands'),
    F.countDistinct('user_session').alias('session_count'),
    
    # Time-based metrics
    F.countDistinct(F.to_date('event_timestamp')).alias('active_days'),
    F.min('event_timestamp').alias('first_event'),
    F.max('event_timestamp').alias('last_event'),
    
    # Total events
    F.count('*').alias('total_events')
)

# Derived features
user_features = user_features.withColumn(
    'purchase_rate',
    F.when(F.col('total_events') > 0, 
           F.round(F.col('purchase_count') / F.col('total_events'), 4)
    ).otherwise(0)
)

user_features = user_features.withColumn(
    'cart_to_purchase_rate',
    F.when(F.col('cart_count') > 0,
           F.round(F.col('purchase_count') / F.col('cart_count'), 4)
    ).otherwise(0)
)

user_features = user_features.withColumn(
    'view_to_cart_rate',
    F.when(F.col('view_count') > 0,
           F.round(F.col('cart_count') / F.col('view_count'), 4)
    ).otherwise(0)
)

# Days since last activity (recency)
user_features = user_features.withColumn(
    'days_active_span',
    F.datediff(F.col('last_event'), F.col('first_event'))
)

# Average events per day
user_features = user_features.withColumn(
    'events_per_day',
    F.when(F.col('active_days') > 0,
           F.round(F.col('total_events') / F.col('active_days'), 2)
    ).otherwise(F.col('total_events'))
)

# Binary flag: cart without purchase (KEY churn indicator)
user_features = user_features.withColumn(
    'cart_without_purchase_flag',
    F.when((F.col('cart_count') > 0) & (F.col('purchase_count') == 0), 1).otherwise(0)
)

# Cache features
user_features.cache()

feature_count = user_features.count()
print(f'✅ Feature engineering complete')
print(f'   Users with features: {feature_count:,}')
print(f'   Total features created: {len(user_features.columns)}')

In [ ]:
# Display feature summary
print('=' * 80)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 80)

print('\n📊 Created features:')
print('\nEvent Counts:')
print('  • view_count, cart_count, remove_from_cart_count, purchase_count')
print('\nFinancial Metrics:')
print('  • total_spent, average_price, max_price_viewed, min_price_viewed')
print('\nDiversity Metrics:')
print('  • unique_products, unique_categories, unique_brands, session_count')
print('\nTime-Based:')
print('  • active_days, first_event, last_event, days_active_span, events_per_day')
print('\nConversion Rates:')
print('  • purchase_rate, cart_to_purchase_rate, view_to_cart_rate')
print('\nChurn Indicators:')
print('  • cart_without_purchase_flag (KEY)')

print('\n' + '=' * 80)
user_features.printSchema()

print('\n📋 Sample features:')
user_features.show(5, truncate=False)

In [ ]:
# Feature statistics
print('=' * 60)
print('FEATURE STATISTICS')
print('=' * 60)

feature_stats = user_features.select(
    F.mean('view_count').alias('avg_views'),
    F.mean('cart_count').alias('avg_carts'),
    F.mean('purchase_count').alias('avg_purchases'),
    F.mean('total_spent').alias('avg_spent'),
    F.mean('purchase_rate').alias('avg_purchase_rate'),
    F.sum('cart_without_purchase_flag').alias('users_cart_no_purchase')
).collect()[0]

print(f'Avg views/user:              {feature_stats.avg_views:>10.2f}')
print(f'Avg carts/user:              {feature_stats.avg_carts:>10.2f}')
print(f'Avg purchases/user:          {feature_stats.avg_purchases:>10.2f}')
print(f'Avg spend/user:              ${feature_stats.avg_spent:>9.2f}')
print(f'Avg purchase rate:           {feature_stats.avg_purchase_rate:>10.4f}')
print(f'\nUsers with cart abandonment: {feature_stats.users_cart_no_purchase:>10,}')
print(f'Abandonment rate:            {(feature_stats.users_cart_no_purchase/feature_count)*100:>9.1f}%')
print('=' * 60)

---
## Layer 6: Churn Risk Classification 🎯

Label users based on behavior patterns

In [ ]:
print('🎯 Applying churn risk logic...')
print('\nChurn Risk Definitions:')
print('  🔴 HIGH:   cart > 0 AND purchase = 0 (abandoned cart)')
print('  🟡 MEDIUM: view > 5 AND cart = 0 (browsing, no action)')
print('  🟢 LOW:    purchase > 0 (converted customer)\n')

# Apply churn logic
user_features_labeled = user_features.withColumn(
    'churn_risk',
    F.when(
        (F.col('cart_count') > 0) & (F.col('purchase_count') == 0),
        'high'
    ).when(
        (F.col('view_count') >= 5) & (F.col('cart_count') == 0),
        'medium'
    ).when(
        F.col('purchase_count') > 0,
        'low'
    ).otherwise('unknown')
)

# Add numeric risk score for ML (0=low, 1=medium, 2=high)
user_features_labeled = user_features_labeled.withColumn(
    'churn_risk_score',
    F.when(F.col('churn_risk') == 'high', 2)
     .when(F.col('churn_risk') == 'medium', 1)
     .when(F.col('churn_risk') == 'low', 0)
     .otherwise(-1)
)

# Add churn reason (explainability)
user_features_labeled = user_features_labeled.withColumn(
    'churn_reason',
    F.when(
        F.col('churn_risk') == 'high',
        F.concat(
            F.lit('Abandoned '),
            F.col('cart_count').cast('string'),
            F.lit(' cart items without purchase')
        )
    ).when(
        F.col('churn_risk') == 'medium',
        F.concat(
            F.lit('Viewed '),
            F.col('view_count').cast('string'),
            F.lit(' products but never added to cart')
        )
    ).when(
        F.col('churn_risk') == 'low',
        F.concat(
            F.lit('Converted customer - '),
            F.col('purchase_count').cast('string'),
            F.lit(' purchases')
        )
    ).otherwise('Insufficient activity')
)

# Cache labeled data
user_features_labeled.cache()

print('✅ Churn labels assigned')
print(f'   Total users labeled: {user_features_labeled.count():,}')

In [ ]:
# Churn distribution
print('=' * 60)
print('CHURN RISK DISTRIBUTION')
print('=' * 60)

churn_dist = (user_features_labeled
              .groupBy('churn_risk')
              .agg(F.count('*').alias('user_count'))
              .withColumn('percentage', 
                         F.round((F.col('user_count') / feature_count) * 100, 2))
              .orderBy(F.desc('user_count')))

churn_dist.show(truncate=False)

# Get specific counts
high_risk = churn_dist.filter(F.col('churn_risk') == 'high').select('user_count').collect()
high_risk_count = high_risk[0]['user_count'] if high_risk else 0

print(f'\n🔴 HIGH RISK users need immediate attention: {high_risk_count:,}')
print(f'   These users have items in cart but haven\'t purchased')
print(f'   → Action: Send discount codes, urgency messages')
print('=' * 60)

In [ ]:
# Sample of each risk category
print('\n📋 Sample users by churn risk:\n')

for risk_level in ['high', 'medium', 'low']:
    print(f'\n{risk_level.upper()} RISK USERS:')
    print('-' * 80)
    (user_features_labeled
     .filter(F.col('churn_risk') == risk_level)
     .select('user_id', 'view_count', 'cart_count', 'purchase_count', 
             'total_spent', 'churn_reason')
     .show(3, truncate=False))

---
## Layer 7: Processed Storage 💾

Save enriched features for ML and analytics

In [ ]:
print('💾 Saving processed data to Parquet...')

# Save user features with churn labels
(user_features_labeled.write
 .mode('overwrite')
 .parquet(f'{PROCESSED_STORAGE}/user_features'))

print(f'✅ Processed data saved')
print(f'   Location: {PROCESSED_STORAGE}/user_features')
print(f'   Format: Parquet')
print(f'   Rows: {user_features_labeled.count():,}')
print(f'   Columns: {len(user_features_labeled.columns)}')

# Verify save
!ls -lh {PROCESSED_STORAGE}/user_features/

# Show size
result = subprocess.run(['du', '-sh', f'{PROCESSED_STORAGE}/user_features'],
                       capture_output=True, text=True)
print(f'\n📊 Storage size: {result.stdout.split()[0]}')

---
## Layer 8: Analytics Tables 📊

Create aggregated tables for dashboards

In [ ]:
print('📊 Creating analytics tables...')
print('   These tables power business intelligence dashboards\n')

# Reload processed data
df_processed = spark.read.parquet(f'{PROCESSED_STORAGE}/user_features')
df_events = spark.read.parquet(f'{RAW_STORAGE}/events')

In [ ]:
# Table 1: Event Type Distribution
print('1️⃣ Event Type Distribution...')

event_type_dist = (df_events
                   .groupBy('event_type')
                   .agg(F.count('*').alias('event_count'))
                   .withColumn('percentage',
                              F.round((F.col('event_count') / df_events.count()) * 100, 2))
                   .orderBy(F.desc('event_count')))

(event_type_dist.write
 .mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/event_type_distribution'))

event_type_dist.show(truncate=False)
print('   ✅ Saved to analytics/event_type_distribution')

In [ ]:
# Table 2: Top Categories
print('\n2️⃣ Top Categories...')

top_categories = (df_events
                  .filter((F.col('category_code').isNotNull()) & 
                         (F.col('category_code') != 'unknown'))
                  .groupBy('category_code')
                  .agg(
                      F.count('*').alias('total_events'),
                      F.countDistinct('user_id').alias('unique_users'),
                      F.sum(F.when(F.col('event_type') == 'purchase', 1).otherwise(0)).alias('purchases'),
                      F.avg('price').alias('avg_price')
                  )
                  .orderBy(F.desc('total_events'))
                  .limit(20))

(top_categories.write
 .mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/top_categories'))

top_categories.show(10, truncate=False)
print('   ✅ Saved to analytics/top_categories')

In [ ]:
# Table 3: Top Brands
print('\n3️⃣ Top Brands...')

top_brands = (df_events
              .filter((F.col('brand').isNotNull()) & (F.col('brand') != 'unknown'))
              .groupBy('brand')
              .agg(
                  F.count('*').alias('total_events'),
                  F.countDistinct('user_id').alias('unique_users'),
                  F.sum(F.when(F.col('event_type') == 'purchase', 1).otherwise(0)).alias('purchases'),
                  F.sum(F.when(F.col('event_type') == 'purchase', F.col('price')).otherwise(0)).alias('total_revenue')
              )
              .orderBy(F.desc('total_revenue'))
              .limit(20))

(top_brands.write
 .mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/top_brands'))

top_brands.show(10, truncate=False)
print('   ✅ Saved to analytics/top_brands')

In [ ]:
# Table 4: Churn Risk Distribution (from processed data)
print('\n4️⃣ Churn Risk Distribution...')

churn_risk_dist = (df_processed
                   .groupBy('churn_risk')
                   .agg(
                       F.count('*').alias('user_count'),
                       F.avg('total_spent').alias('avg_spent'),
                       F.avg('purchase_count').alias('avg_purchases'),
                       F.avg('cart_count').alias('avg_carts')
                   )
                   .withColumn('percentage',
                              F.round((F.col('user_count') / df_processed.count()) * 100, 2))
                   .orderBy(F.desc('user_count')))

(churn_risk_dist.write
 .mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/churn_risk_distribution'))

churn_risk_dist.show(truncate=False)
print('   ✅ Saved to analytics/churn_risk_distribution')

In [ ]:
# Table 5: Daily Activity Trend
print('\n5️⃣ Daily Activity Trend...')

# Re-add event_timestamp if needed
df_events_ts = df_events.withColumn(
    'event_timestamp',
    F.to_timestamp('event_time', 'yyyy-MM-dd HH:mm:ss z')
)

# If format doesn't match
if df_events_ts.filter(F.col('event_timestamp').isNull()).count() > 0:
    df_events_ts = df_events.withColumn(
        'event_timestamp',
        F.to_timestamp('event_time')
    )

daily_activity = (df_events_ts
                  .filter(F.col('event_timestamp').isNotNull())
                  .withColumn('date', F.to_date('event_timestamp'))
                  .groupBy('date')
                  .agg(
                      F.count('*').alias('total_events'),
                      F.countDistinct('user_id').alias('active_users'),
                      F.sum(F.when(F.col('event_type') == 'view', 1).otherwise(0)).alias('views'),
                      F.sum(F.when(F.col('event_type') == 'cart', 1).otherwise(0)).alias('carts'),
                      F.sum(F.when(F.col('event_type') == 'purchase', 1).otherwise(0)).alias('purchases'),
                      F.sum(F.when(F.col('event_type') == 'purchase', F.col('price')).otherwise(0)).alias('revenue')
                  )
                  .orderBy('date'))

(daily_activity.write
 .mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/daily_activity'))

daily_activity.show(10, truncate=False)
print('   ✅ Saved to analytics/daily_activity')

In [ ]:
# Table 6: Cart Abandonment Rate
print('\n6️⃣ Cart Abandonment Analysis...')

cart_abandonment = df_processed.select(
    F.count('*').alias('total_users'),
    F.sum(F.when(F.col('cart_count') > 0, 1).otherwise(0)).alias('users_with_cart'),
    F.sum(F.when((F.col('cart_count') > 0) & (F.col('purchase_count') == 0), 1).otherwise(0)).alias('cart_abandoned'),
    F.sum(F.when((F.col('cart_count') > 0) & (F.col('purchase_count') > 0), 1).otherwise(0)).alias('cart_converted')
).withColumn(
    'abandonment_rate',
    F.when(F.col('users_with_cart') > 0,
           F.round((F.col('cart_abandoned') / F.col('users_with_cart')) * 100, 2)
    ).otherwise(0)
).withColumn(
    'conversion_rate',
    F.when(F.col('users_with_cart') > 0,
           F.round((F.col('cart_converted') / F.col('users_with_cart')) * 100, 2)
    ).otherwise(0)
)

(cart_abandonment.write
 .mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/cart_abandonment'))

cart_abandonment.show(truncate=False)
print('   ✅ Saved to analytics/cart_abandonment')

print('\n' + '=' * 60)
print('ANALYTICS TABLES CREATED')
print('=' * 60)
print('  1. event_type_distribution')
print('  2. top_categories')
print('  3. top_brands')
print('  4. churn_risk_distribution')
print('  5. daily_activity')
print('  6. cart_abandonment')
print('=' * 60)

---
## Layer 9: Machine Learning 🤖

Train models to predict churn risk

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

print('🤖 Preparing data for Machine Learning...')

# Load processed features
ml_data = spark.read.parquet(f'{PROCESSED_STORAGE}/user_features')

# Select features for ML (numeric only)
feature_cols = [
    'view_count', 'cart_count', 'remove_from_cart_count', 'purchase_count',
    'total_spent', 'average_price', 'max_price_viewed', 'min_price_viewed',
    'unique_products', 'unique_categories', 'unique_brands', 'session_count',
    'active_days', 'total_events', 'purchase_rate', 'cart_to_purchase_rate',
    'view_to_cart_rate', 'days_active_span', 'events_per_day',
    'cart_without_purchase_flag'
]

print(f'   Features selected: {len(feature_cols)}')
print(f'   Target variable: churn_risk_score')

In [ ]:
# Prepare ML dataset
# Filter out 'unknown' risk and ensure we have labels
ml_data_clean = (ml_data
                 .filter((F.col('churn_risk_score') >= 0) & 
                        (F.col('churn_risk') != 'unknown'))
                 .select(feature_cols + ['churn_risk_score', 'churn_risk']))

# Fill any remaining nulls with 0 (for ML algorithms)
ml_data_clean = ml_data_clean.fillna(0)

# Check class distribution
print('\n📊 Target variable distribution:')
ml_data_clean.groupBy('churn_risk', 'churn_risk_score').count().orderBy('churn_risk_score').show()

total_ml_samples = ml_data_clean.count()
print(f'\nTotal samples for ML: {total_ml_samples:,}')

In [ ]:
# Create feature vector
print('\n🔧 Creating feature vectors...')

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features_raw'
)

# Scale features for better model performance
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withStd=True,
    withMean=True
)

# Apply transformations
ml_data_vec = assembler.transform(ml_data_clean)
scaler_model = scaler.fit(ml_data_vec)
ml_data_scaled = scaler_model.transform(ml_data_vec)

# Rename target for ML
ml_data_final = ml_data_scaled.withColumnRenamed('churn_risk_score', 'label')

print('✅ Feature vectors created and scaled')
print(f'   Feature vector size: {len(feature_cols)}')

# Split data: 80% train, 20% test
train_data, test_data = ml_data_final.randomSplit([0.8, 0.2], seed=42)

train_count = train_data.count()
test_count = test_data.count()

print(f'\n📊 Train/Test split:')
print(f'   Training samples: {train_count:,} (80%)')
print(f'   Test samples:     {test_count:,} (20%)')

In [ ]:
# Model 1: Logistic Regression
print('\n' + '=' * 60)
print('MODEL 1: Logistic Regression')
print('=' * 60)

lr = LogisticRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    regParam=0.01
)

print('Training Logistic Regression...')
lr_model = lr.fit(train_data)

print('✅ Training complete')
print('\nMaking predictions on test set...')
lr_predictions = lr_model.transform(test_data)

# Evaluate
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1'
)
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='weightedPrecision'
)
evaluator_recall = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='weightedRecall'
)

lr_accuracy = evaluator_acc.evaluate(lr_predictions)
lr_f1 = evaluator_f1.evaluate(lr_predictions)
lr_precision = evaluator_precision.evaluate(lr_predictions)
lr_recall = evaluator_recall.evaluate(lr_predictions)

print(f'\nLogistic Regression Results:')
print(f'  Accuracy:  {lr_accuracy:.4f}')
print(f'  F1 Score:  {lr_f1:.4f}')
print(f'  Precision: {lr_precision:.4f}')
print(f'  Recall:    {lr_recall:.4f}')

In [ ]:
# Model 2: Decision Tree
print('\n' + '=' * 60)
print('MODEL 2: Decision Tree')
print('=' * 60)

dt = DecisionTreeClassifier(
    featuresCol='features',
    labelCol='label',
    maxDepth=10,
    minInstancesPerNode=10
)

print('Training Decision Tree...')
dt_model = dt.fit(train_data)

print('✅ Training complete')
print('\nMaking predictions on test set...')
dt_predictions = dt_model.transform(test_data)

dt_accuracy = evaluator_acc.evaluate(dt_predictions)
dt_f1 = evaluator_f1.evaluate(dt_predictions)
dt_precision = evaluator_precision.evaluate(dt_predictions)
dt_recall = evaluator_recall.evaluate(dt_predictions)

print(f'\nDecision Tree Results:')
print(f'  Accuracy:  {dt_accuracy:.4f}')
print(f'  F1 Score:  {dt_f1:.4f}')
print(f'  Precision: {dt_precision:.4f}')
print(f'  Recall:    {dt_recall:.4f}')

# Feature importance
print('\n📊 Top 10 Important Features:')
feature_importance = [(feature_cols[i], float(dt_model.featureImportances[i]))
                     for i in range(len(feature_cols))]
feature_importance_sorted = sorted(feature_importance, key=lambda x: x[1], reverse=True)

for feat, importance in feature_importance_sorted[:10]:
    print(f'  {feat:30s} {importance:.4f}')

In [ ]:
# Model 3: Random Forest
print('\n' + '=' * 60)
print('MODEL 3: Random Forest')
print('=' * 60)

rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    numTrees=50,
    maxDepth=10,
    minInstancesPerNode=10,
    seed=42
)

print('Training Random Forest (50 trees)...')
rf_model = rf.fit(train_data)

print('✅ Training complete')
print('\nMaking predictions on test set...')
rf_predictions = rf_model.transform(test_data)

rf_accuracy = evaluator_acc.evaluate(rf_predictions)
rf_f1 = evaluator_f1.evaluate(rf_predictions)
rf_precision = evaluator_precision.evaluate(rf_predictions)
rf_recall = evaluator_recall.evaluate(rf_predictions)

print(f'\nRandom Forest Results:')
print(f'  Accuracy:  {rf_accuracy:.4f}')
print(f'  F1 Score:  {rf_f1:.4f}')
print(f'  Precision: {rf_precision:.4f}')
print(f'  Recall:    {rf_recall:.4f}')

# Feature importance
print('\n📊 Top 10 Important Features (Random Forest):')
rf_feature_importance = [(feature_cols[i], float(rf_model.featureImportances[i]))
                        for i in range(len(feature_cols))]
rf_feature_importance_sorted = sorted(rf_feature_importance, key=lambda x: x[1], reverse=True)

for feat, importance in rf_feature_importance_sorted[:10]:
    print(f'  {feat:30s} {importance:.4f}')

In [ ]:
# Model comparison
print('\n' + '=' * 80)
print('MODEL COMPARISON')
print('=' * 80)

import pandas as pd

comparison_df = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': lr_accuracy, 'F1': lr_f1, 
     'Precision': lr_precision, 'Recall': lr_recall},
    {'Model': 'Decision Tree', 'Accuracy': dt_accuracy, 'F1': dt_f1,
     'Precision': dt_precision, 'Recall': dt_recall},
    {'Model': 'Random Forest', 'Accuracy': rf_accuracy, 'F1': rf_f1,
     'Precision': rf_precision, 'Recall': rf_recall}
])

print(comparison_df.to_string(index=False))

# Find best model
best_model_idx = comparison_df['F1'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_f1 = comparison_df.loc[best_model_idx, 'F1']

print(f'\n🏆 BEST MODEL: {best_model_name} (F1 = {best_f1:.4f})')
print('=' * 80)

# Save comparison to Spark DataFrame for export
comparison_spark = spark.createDataFrame(comparison_df)
(comparison_spark.write
 .mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/model_comparison'))

print('\n✅ Model comparison saved to analytics/model_comparison')

In [ ]:
# Save best model
print(f'\n💾 Saving best model ({best_model_name})...')

if best_model_name == 'Random Forest':
    best_model = rf_model
    best_predictions = rf_predictions
elif best_model_name == 'Decision Tree':
    best_model = dt_model
    best_predictions = dt_predictions
else:
    best_model = lr_model
    best_predictions = lr_predictions

best_model.write().overwrite().save(f'{ML_STORAGE}/best_churn_model')
print(f'✅ Model saved to {ML_STORAGE}/best_churn_model')

---
## Layer 10: Decision Engine 🎯

Generate actionable recommendations

In [ ]:
print('🎯 Applying Decision Engine...')
print('   Generating personalized retention actions\n')

# Use best model predictions
# Map prediction back to risk level
final_decisions = best_predictions.withColumn(
    'predicted_risk',
    F.when(F.col('prediction') == 2, 'high')
     .when(F.col('prediction') == 1, 'medium')
     .when(F.col('prediction') == 0, 'low')
     .otherwise('unknown')
)

# Add recommended action based on predicted risk
final_decisions = final_decisions.withColumn(
    'recommended_action',
    F.when(
        F.col('predicted_risk') == 'high',
        'Send 20% discount code + urgent reminder (cart expires in 24h)'
    ).when(
        F.col('predicted_risk') == 'medium',
        'Send personalized product recommendations + 10% first-purchase discount'
    ).when(
        F.col('predicted_risk') == 'low',
        'No immediate action - maintain engagement with newsletters'
    ).otherwise('Collect more behavioral data')
)

# Add action priority
final_decisions = final_decisions.withColumn(
    'priority',
    F.when(F.col('predicted_risk') == 'high', 1)
     .when(F.col('predicted_risk') == 'medium', 2)
     .when(F.col('predicted_risk') == 'low', 3)
     .otherwise(4)
)

# Select final output columns
final_output = final_decisions.select(
    F.col('user_id'),
    F.col('churn_risk').alias('actual_risk'),
    F.col('predicted_risk'),
    F.col('churn_reason').alias('reason'),
    F.col('recommended_action'),
    F.col('priority'),
    F.col('total_spent'),
    F.col('cart_count'),
    F.col('purchase_count'),
    F.col('view_count')
).orderBy('priority', F.desc('total_spent'))

print('✅ Decision engine applied')
print(f'   Total decisions generated: {final_output.count():,}')

In [ ]:
# Show sample decisions for each priority
print('\n' + '=' * 80)
print('SAMPLE RETENTION ACTIONS')
print('=' * 80)

for priority_level in [1, 2, 3]:
    priority_name = {1: 'HIGH PRIORITY', 2: 'MEDIUM PRIORITY', 3: 'LOW PRIORITY'}[priority_level]
    print(f'\n{priority_name}:')
    print('-' * 80)
    (final_output
     .filter(F.col('priority') == priority_level)
     .select('user_id', 'predicted_risk', 'recommended_action', 'cart_count', 'total_spent')
     .show(3, truncate=60))

In [ ]:
# Action distribution
print('=' * 60)
print('ACTION DISTRIBUTION')
print('=' * 60)

action_dist = (final_output
               .groupBy('predicted_risk', 'recommended_action')
               .count()
               .orderBy('predicted_risk'))

action_dist.show(truncate=False)

# Calculate potential revenue impact
high_risk_users = final_output.filter(F.col('predicted_risk') == 'high').count()
avg_cart_value = final_output.filter(F.col('predicted_risk') == 'high').agg(
    F.avg('total_spent')
).collect()[0][0] or 0

# Assume 15% conversion rate from discount campaign
potential_conversions = high_risk_users * 0.15
potential_revenue = potential_conversions * avg_cart_value

print(f'\n💰 POTENTIAL BUSINESS IMPACT:')
print(f'   High-risk users targeted: {high_risk_users:,}')
print(f'   Expected conversion rate: 15%')
print(f'   Potential conversions:    {potential_conversions:,.0f}')
print(f'   Avg cart value:           ${avg_cart_value:,.2f}')
print(f'   Estimated revenue gain:   ${potential_revenue:,.2f}')
print('=' * 60)

---
## Layer 11: Export Layer 📤

Export CSV files for dashboards and business teams

In [ ]:
print('📤 Exporting data to CSV for dashboards...\n')

# Export 1: Churn Risk Distribution
print('1️⃣ Churn risk distribution...')
churn_risk_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/churn_risk_distribution')
(churn_risk_export.coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/churn_risk_distribution'))
print('   ✅ Exported')

# Export 2: Event Type Distribution
print('\n2️⃣ Event type distribution...')
event_type_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/event_type_distribution')
(event_type_export.coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/event_type_distribution'))
print('   ✅ Exported')

# Export 3: Top Categories
print('\n3️⃣ Top categories...')
top_cat_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/top_categories')
(top_cat_export.coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/top_categories'))
print('   ✅ Exported')

# Export 4: Top Brands
print('\n4️⃣ Top brands...')
top_brands_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/top_brands')
(top_brands_export.coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/top_brands'))
print('   ✅ Exported')

# Export 5: Daily Activity
print('\n5️⃣ Daily activity trend...')
daily_act_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/daily_activity')
(daily_act_export.coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/daily_activity'))
print('   ✅ Exported')

# Export 6: Model Comparison
print('\n6️⃣ Model comparison...')
model_comp_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/model_comparison')
(model_comp_export.coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/model_comparison'))
print('   ✅ Exported')

# Export 7: Final User Decisions (Top 1000 priority users)
print('\n7️⃣ Final user decisions (top 1000 priority)...')
(final_output.limit(1000).coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/final_user_decisions'))
print('   ✅ Exported')

print('\n' + '=' * 60)
print('ALL EXPORTS COMPLETE')
print('=' * 60)
print(f'Location: {EXPORT_STORAGE}')
print('Files:')
print('  1. churn_risk_distribution.csv')
print('  2. event_type_distribution.csv')
print('  3. top_categories.csv')
print('  4. top_brands.csv')
print('  5. daily_activity.csv')
print('  6. model_comparison.csv')
print('  7. final_user_decisions.csv (top 1000)')
print('=' * 60)

In [ ]:
# List all exported CSV files
print('\n📂 Verifying exports...')
!find {EXPORT_STORAGE} -name '*.csv' -type f

---
## 🎉 Pipeline Complete!

### Summary

✅ **11 Layers Implemented:**
1. Data Ingestion (PySpark)
2. Raw Storage (Parquet + Partitioning)
3. Data Quality Check
4. Data Cleaning
5. Feature Engineering (20+ features)
6. Churn Risk Classification
7. Processed Storage
8. Analytics Tables (6 tables)
9. Machine Learning (3 models)
10. Decision Engine (Retention actions)
11. CSV Exports (7 files)

### Business Value

This pipeline enables:
- **Proactive Retention**: Identify at-risk customers before they churn
- **Personalized Actions**: Tailored discounts and reminders
- **Revenue Recovery**: Convert abandoned carts into sales
- **Data-Driven Decisions**: ML-powered insights at scale

### Scalability

The architecture scales from:
- **Development**: 500K rows (this notebook)
- **Production**: 100M+ rows (same code, just remove sampling)

Change required: Remove `.sample(fraction=0.1)` in Layer 1

---

### Next Steps

1. **Deploy to production** (Databricks, EMR, or Dataproc)
2. **Schedule daily runs** (Airflow, cron)
3. **Integrate with CRM** (send actions to marketing team)
4. **Build dashboard** (Tableau, Power BI)
5. **Monitor model drift** (retrain monthly)

---